# Assignment: My Data is a Mess. Can I Trust Any of It?

**Course:** Quantitative Data Analysis (INFOMQUDA)  
**Environment:** Google Colab (CPU is sufficient)  
**Datasets:** Synthetic heart-rate data (Sections 1–2) · Gapminder (Sections 3–6)

---

## Requirements for this practical

For the first tutorial session, you are expected to complete **Sections 1–2**.

The remaining sections will be made available before the next tutorial.

In these practicals, you should be able to:

1. understand what each code cell is doing;
2. change parameters and explain how the results change;
3. reproduce the main parts of the code yourself;
4. answer the questions at the end of each section;
5. ask questions during the practical if anything is unclear.

Questions asked during the practical will be noted together with attendance.

In every section, the questions are placed **at the end of the section**.


## Overview

Before you can trust any analysis, you need to trust your data. This assignment works through the most common ways data can mislead you — and what to do about it.

You will:

1. **See** how noise can hide a real signal — and how denoising recovers it  
2. **Understand** outliers: when to remove them, and what they reveal about your sample  
3. **Explore** data distributions and what "normal" really means  
4. **Transform** data and learn when that helps — and when it creates new problems
5. **Question** when to trust a significant relationship and when to dig deeper
6. **Test** hypotheses correctly, including what to do when you test many at once  
7. **Sample** wisely and understand what your results can and cannot generalise to  

---

## How to use this notebook

- **Explanatory cells** introduce each concept and explain *why* it matters.  
- **Code cells** contain working implementations — read them before running.  
- Cells marked **STUDENT TASK** ask you to adjust parameters or input your own code.  
- Questions marked **Q** should be answered in the notebook (edit the `> Your answer here` box).  

> **Colab note:** Run cells top-to-bottom. If your runtime disconnects, restart from revant upper Sections.

---

## Table of Contents

| Section | Topic |
|---------|-------|
| [0. Setup](#setup) | Packages, imports, reproducibility |
| [1. Noise](#noise) | When noise hides a real relationship |
| [2. Outliers](#outliers) | When outliers change real results |
| [3. Distributions](#distributions) | What does the data actually look like? |
| [4. Transformations](#transformations) | Useful — but dangerous |
| [5. Misleading relationships](#misleading) | Can we still be missing something? |
| [5. Hypothesis testing](#testing) | Significance tests and multiple comparisons |
| [6. Sampling](#sampling) | Who is in your data, and who is not? |


---
## Section 0: Setup <a id='setup'></a>

We install and import everything once here. All subsequent sections assume these have run.

**Packages used:**

| Package | Purpose |
|---------|---------|
| `numpy`, `pandas` | Arrays and data frames |
| `matplotlib`, `seaborn` | Plotting |
| `scipy.stats` | Statistical tests |
| `sklearn` | Preprocessing, models, splits, metrics |
| `torch` | Neural network (refresher in this section) |


In [ ]:
# ── 0.1  Install packages ────────────────────────────────────────────────────
# Run this first. Output is suppressed for readability.
!pip install -q numpy pandas matplotlib seaborn scipy scikit-learn torch statsmodels > /dev/null 2>&1
print("Packages ready.")


In [ ]:
# ── 0.2  Imports and global settings ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 36
np.random.seed(SEED)

# Plot style — clean, minimal
plt.rcParams.update({
    "figure.dpi":        110,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
})

print("Imports done.")


### Scikit-learn refresher

scikit-learn is the standard Python library for machine learning and data preprocessing.  

Three ideas are worth recalling before we start, because we will use all three later.

**The fit / transform pattern**
```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)               # learns mean and std from training data only
X_scaled = scaler.transform(X_train)   # applies the learned transformation
```
Always `fit` on training data and `transform` everything else with the *same fitted object*. Fitting on the full dataset before splitting is one of the most common data-leakage mistakes.

**train_test_split**
```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```
`random_state` makes the split reproducible. For time-series or repeated-measures data, random splitting is *wrong* — we return to this in Section 6.

**A simple pipeline**
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression()),
])
pipe.fit(X_train, y_train)
pipe.predict(X_test)
```
Pipelines chain preprocessing and modelling so fitting and transforming always happen in the correct order, even inside cross-validation.


In [ ]:
# CODE HERE START
# Task:
# 1. Create x values from 0 to 19.
# 2. Create y values using approximately: y = 2*x + noise.
# 3. Fit a LinearRegression model.
# 4. Print the slope.


# CODE HERE END

### PyTorch refresher

PyTorch is the standard deep-learning library. We use a small network in Section 6 to illustrate per-person data splitting. Three concepts are worth remembering.

**Tensors** — PyTorch's equivalent of NumPy arrays, supporting GPU and automatic differentiation:
```python
import torch
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])    # shape (2, 2)
x = torch.from_numpy(numpy_array).float()      # convert from NumPy
```

**A minimal neural network**
```python
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)   # output shape: (batch,)
```

**A minimal training loop** — these four steps are always in this order:
```python
model     = SimpleNet(input_dim=10)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(100):
    model.train()
    pred  = model(X_tensor)         # 1. forward pass
    loss  = criterion(pred, y_tensor)   # 2. compute loss
    optimizer.zero_grad()           # 3. clear old gradients
    loss.backward()                 # 4. backpropagate
    optimizer.step()                # 5. update weights
```


**Device** — For this practical, we use the **CPU**. CUDA/GPU is not needed for this notebook. To use CUDA, you would need to select a GPU runtime in the notebook settings and change the device line in the code. This may use more computing credits and is unnecessary for this practical.

In [ ]:
# ── 0.3  Quick PyTorch test ────────────────────────────────────────────
import torch
import torch.nn as nn

DEVICE = device = torch.device("cpu")
print(f"PyTorch {torch.__version__} · device: {DEVICE}")

class SimpleNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# CODE HERE START
# 1. define simplenet
# 2. create a small dataset
# 3. apply simple net to this data
# 4. print the shape of the data and output


# CODE HERE END


---
# PART I. Synthetic Heart Rate Dataset - Does exam stress lead to high heart rate?

We generate synthetic heart-rate (HR) data simulating students just before an exam. We assume that **low stress** before an exam leads to lower HR and **high stress** to higher HR.

Heart rate is often measured in **BPM**, which means **beats per minute**.

Important: this is **not an ECG simulation**.

An ECG signal shows electrical activity of the heart and contains sharp R-peaks. Here, we are already working with a simplified heart-rate time series: each value is an estimated HR in BPM.

So this notebook simulates something closer to:

> “The wearable estimated that the person’s HR was 72 BPM at this time point.”

not:

> “This is the raw electrical waveform of the heart.”

The simulation is intentionally simple and limited. Its purpose is teaching, not physiological realism.

---
## Section 1: When Noise Hides a Real Relationship <a id='noise'></a>

High stress raises HR by a fixed amount — there *is* a real difference in the data. But when we add enough measurement noise, that difference can become invisible to a statistical test.

This section shows:
- How signal-to-noise ratio affects whether you can detect a true effect  
- Two denoising strategies and how well each recovers the original signal  
- Why denoising is powerful but can also create artefacts if misapplied  

**Synthetic data** — We simulate HR using a sine wave:

\[
\text{HR drift}(t) = A \sin(2\pi f t)
\]

where:

- \(A\) is the amplitude of the drift;
- \(f\) is the frequency;
- \(t\) is time.

We also simulate several types of variation in the data:
1. Between-participant variation: we draw from a normal distribution with mean of true HR and standad deviation of PARTICIPANT_VARIABILITY
2. Within participant variation: HR_DRIFT_AMPLITUDE, HR_DRIFT_FREQUENCY
3. Small random time-point noise: independent fluctuation at every measurement, drawn from a normal distribution with standard deviation of TIME_POINT_NOISE BPM
4. Artefact bursts: simulating movement, slipped sensor, or bad contact. Each participants gets between N_ARTIFACT_MIN and N_ARTIFACT_MAX episodes of amplitude ARTIFACT_BURST, each lasting ARTIFACT_MIN_LENGTH to ARTIFACT_MAX_LENGTH seconds.


In [ ]:
# ── 1.1  Synthetic HR data generator ─────────────────────────────────────────

# STUDENT TASK: adjust the parameters below and re-run the cells

N_SAMPLES   = 20      # participants per condition
HR_LOW      = 70      # group mean HR, low stress
HR_HIGH     = 75      # group mean HR, high stress

DURATION    = 5.0     # in minutes
FS          = 60      # number of measurements per minute
PARTICIPANT = 5       # participant to inspect

PARTICIPANT_VARIABILITY = 5
HR_DRIFT_AMPLITUDE = 3
HR_DRIFT_FREQUENCY = 0.1

N_ARTIFACT_MIN = 1
N_ARTIFACT_MAX = 4        
ARTIFACT_MIN_LENGTH = 20  # seconds
ARTIFACT_MAX_LENGTH = 80  # seconds
TIME_POINT_NOISE = 5
ARTIFACT_BURST = 25      # in BPM

# ─────────────────────────────────────────────────────────────────────────────

t = np.linspace(0, DURATION, int(DURATION * FS))
T = len(t)

def make_hr_signals(n_samples, seed=369):
    np.random.seed(seed)
    signals = {}
    for condition, true_hr in [("low", HR_LOW), ("high", HR_HIGH)]:

        # CODE HERE START
        # 1. Between-participant variation: give each of the n_samples participants
        #    their own baseline HR, drawn from a normal distribution centred on
        #    true_hr with a standard deviation of PARTICIPANT_VARIABILITY.
        # 2. Within-participant drift: build a slow sinusoid over t with amplitude
        #    HR_DRIFT_AMPLITUDE and frequency HR_DRIFT_FREQUENCY based on the function
        #    given above.

        # between_p =
        # within_p =

        # CODE HERE END

        clean     = between_p[:, None] + within_p[None, :]
        noisy     = clean.copy()

        # 3. Small random noise at every single measurement.
        wobble = np.random.randn(n_samples, T) * TIME_POINT_NOISE

        # 4. Artefact bursts: sensor reads too high.
        artefacts = np.zeros((n_samples, T))
        for i in range(n_samples):
            for _ in range(np.random.randint(N_ARTIFACT_MIN, N_ARTIFACT_MAX)):
                length = np.random.randint(ARTIFACT_MIN_LENGTH, ARTIFACT_MAX_LENGTH)       
                start  = np.random.randint(0, T - length)
                mag    = np.abs(np.random.randn()) * ARTIFACT_BURST  
                artefacts[i, start:start + length] += mag


        noisy = clean + wobble + artefacts

        signals[condition] = (clean, noisy)
    return signals

signals = make_hr_signals(N_SAMPLES)
clean_low,  noisy_low  = signals["low"]
clean_high, noisy_high = signals["high"]

print(f"Generated {N_SAMPLES} participants per condition")
print(f"Noise level:        {ARTIFACT_BURST} BPM")
print(f"True HR difference: {HR_HIGH - HR_LOW} BPM")
print(f"Signal shape:       {clean_low.shape}  (participants × time-points)")

In [ ]:
# ── 1.2  Single-participant view: clean vs noisy ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)

for ax, data, label in zip(
        axes,
        [clean_low, noisy_low],
        ["Clean signal — participant 5, low stress",
         "Noisy signal — participant 5, low stress"]):
    ax.plot(t, data[PARTICIPANT-1], lw=1.2, color="steelblue")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Time (minutes)")
    ax.set_ylabel("Heart rate (BPM)")

plt.tight_layout()
plt.show()


In [ ]:
# ── 1.3  Group means: Can you see the difference with noise? ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (cl, ch), label in zip(
        axes,
        [(clean_low, clean_high), (noisy_low, noisy_high)],
        ["Clean data", "Noisy data"]):
    for sig, col in zip([cl, ch], ["steelblue", "tomato"]):
        for row in sig:
            ax.plot(t, row, alpha=0.15, lw=0.6, color=col)
    ax.plot(t, cl.mean(0), color="steelblue", lw=2, label="Low stress mean")
    ax.plot(t, ch.mean(0), color="tomato",    lw=2, label="High stress mean")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Time (minutes)")
    ax.set_ylabel("Heart rate (BPM)")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── 1.4  Statistical test: does the group difference survive noise? ────────────
# We have N participants per condition, each with a full time series.
# The t-test assumes independent observations.
#
# Hence, we summarise each participant's recording into ONE number,
# then compare those numbers across groups with a t-test.
# The summary you choose can determine what effect you can detect.

def summarise(data):
    """data shape: (n_participants, n_timepoints)"""
    mean_hr      = data.mean(axis=1)                  # average level
    peak_hr      = data.max(axis=1)                   # highest point reached
    return mean_hr, peak_hr

low_clean_mean,  low_clean_peak  = summarise(clean_low)
high_clean_mean, high_clean_peak = summarise(clean_high)
low_noisy_mean,  low_noisy_peak  = summarise(noisy_low)
high_noisy_mean, high_noisy_peak = summarise(noisy_high)

print(f"{'Summary':<18}  {'clean p':>9}  {'noisy p':>9}")
print("─" * 40)
for label, cl, ch, nl, nh in [
    ("Mean HR",       low_clean_mean, high_clean_mean, low_noisy_mean, high_noisy_mean),
    ("Peak HR",       low_clean_peak, high_clean_peak, low_noisy_peak, high_noisy_peak)
]:
    _, p_clean = stats.ttest_ind(cl, ch)
    _, p_noisy = stats.ttest_ind(nl, nh)
    print(f"{label:<18}  {p_clean:>9.4f}  {p_noisy:>9.4f}")


### 1.5 Denoising strategies

Denoising means trying to recover the underlying signal from noisy measurements.

Different methods solve different problems, for example:

| Method | What it does | Works best for |
|---|---|---|
| Rolling mean | Averages neighbouring values | small random noise |
| Convolution smoother | Same idea as rolling mean, written as a filter | small random noise |
| Sensor clean-up | Removes implausible readings and interpolates across them | sensor artefacts |

For more background see: https://www.dspguide.com/ch15.htm

In [ ]:
# ── 1.5  Two denoising methods ─────────────────────────────────────────────
import torch
import torch.nn as nn

# STUDENT TASK: try different window sizes and thresholds
ROLLING_WINDOW = 15    # seconds
MAD_THRESHOLD  = 2.5   # how many robust SDs before a point counts as artefact

# ── Method 1: Rolling mean ────────────────────────────────────────────────────
# Replaces each timepoint with the average of its neighbours.
def rolling_mean(signal, window=ROLLING_WINDOW):
    # CODE HERE START
    # 1. define rolling mean denoising: replace each timepoint with the mean of the surrounding `window` points.


    # CODE HERE END

# ── Method 2: Artefact rejection ─────────────────────────────────────────────
# Detects implausible timepoints and replaces them with the participant median.
def artefact_rejection(signal, threshold=MAD_THRESHOLD):
    med  = np.median(signal)

    # MAD = median absolute deviation: a spread measure that ignores outliers,
    # unlike the standard deviation, which the artefacts themselves would inflate.
    mad  = np.median(np.abs(signal - med))

    # For normally distributed data, MAD * 1.4826 equals the standard deviation.
    # Multiplying by that constant lets us read the threshold as "2.5 SDs",
    # but computed in a way the artefacts cannot distort.
    good = np.abs(signal - med) < threshold * mad * 1.4826 # define good signal

    cleaned = signal.copy()
    cleaned[~good] = np.interp(np.flatnonzero(~good), np.flatnonzero(good), signal[good]) # replace bad timepoints with participant median
    return cleaned

DENOISING = {
    "Rolling mean":       rolling_mean,
    "Artefact rejection": artefact_rejection,
}

def apply_smoother(data_array, fn):
    return np.stack([fn(row) for row in data_array])

denoised = {
    name: (apply_smoother(noisy_low, fn), apply_smoother(noisy_high, fn))
    for name, fn in DENOISING.items()
}

print("All denoising methods applied.")


In [ ]:
# ── 1.6  Side-by-side: original · noisy · three denoising methods ────────────
panels = (
    [("Original clean data", clean_low,  clean_high),
     ("Noisy data",          noisy_low,  noisy_high)] +
    [(name, dl, dh) for name, (dl, dh) in denoised.items()]
)

fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharey=True)

for ax, (title, dl, dh) in zip(axes.flat, panels):
    for sig, col in zip([dl, dh], ["steelblue", "tomato"]):
        for row in sig:
            ax.plot(t, row, alpha=0.12, lw=0.5, color=col)
    ax.plot(t, dl.mean(0), color="steelblue", lw=2, label="Low stress mean")
    ax.plot(t, dh.mean(0), color="tomato",    lw=2, label="High stress mean")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("HR (BPM)")

axes[0, 0].legend(fontsize=8)
plt.suptitle("Original · Noisy · Two denoising methods", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 1.7  t-test: which method recovers the real effect? ──────────────────────
_, p_clean = stats.ttest_ind(clean_low.mean(1), clean_high.mean(1))

print(f"{'Method':<25}  {'clean p':>9}  {'denoised p':>11}  {'recovered?':>12}")
print("─" * 63)

for name, (dl, dh) in denoised.items():
    _, p_denoised = stats.ttest_ind(dl.mean(1), dh.mean(1))
    recovered = "Yes" if p_denoised < 0.05 else "No"
    print(f"{name:<25}  {p_clean:>9.4f}  {p_denoised:>11.4f}")

**Q1 —**

a. Increase/Decrease `ARTIFACT_BURST` and other noise variables. What happens to the visual separation?

b. Change `N_SAMPLES`. Does a larger sample size help even when noise level is high?

c. Which of the two denoising methods recovers the group means in your experiment? Try to explain intuitively why.

d. Describe a scenario where applying a smoother would *create* a false signal that was never present in the raw data.

> *Your answer here*


---
## Section 2: When Outliers Change Real Results <a id='outliers'></a>

Outliers come in different types, and the right decision for each can be different:

| Type | Example | Right action |
|------|---------|-------------|
| **Measurement error** | Sensor glitch records 200 BPM | Remove — not a real observation |
| **Real but rare in our data** | Elderly participant with much lower HR | Think carefully — removing it changes your population |

This section shows:
- How adding both types of outliers changes the statistical conclusions  
- How to identify outliers  
- How to remove outliers


In [ ]:
# ── 2.1  Add outliers to the groups ──────────────────────────────────────────

# STUDENT TASK: try changing N_ELDERLY and N_SENSOR and re-run.

N_ELDERLY = 3     # elderly participants in high-stress group
N_SENSOR  = 3     # sensor failures happening in low-stress group
PARTICIPANT_VARIABILITY = 5

# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(SEED)

# Reuse clean data from Section 1
base_low  = clean_low.mean(axis=1)
base_high = clean_high.mean(axis=1)

# Elderly: real people, genuinely lower HR due to age.
# In our dataset, they "happen to" all have high-stress before the exam.
elderly_high  = 60 + np.random.randn(N_ELDERLY) * PARTICIPANT_VARIABILITY

# Sensor errors: physiologically impossible values from a broken device (200+ BPM).
# In our dataset, the faulty sensor "happened to be" worn only by low-stress individuals.
sensor_low    = np.random.uniform(200, 230, size=N_SENSOR)

# add the outliers to the clean data
high_with_elderly = np.concatenate([base_high, elderly_high])
low_with_sensor   = np.concatenate([base_low,  sensor_low])

DATASETS = {
    "Original":            (base_low,         base_high),
    "Elderly added":       (base_low,          high_with_elderly),
    "Sensor errors added": (low_with_sensor,   base_high),
    "Both added":          (low_with_sensor,     high_with_elderly),
}

print(f"{'Dataset':<25}  {'n (low/high)':>13}  {'t':>6}  {'p':>8}  {'Sig?'}")
print("─" * 65)
for name, (lo, hi) in DATASETS.items():
    t_val, p_val = stats.ttest_ind(lo, hi)
    sig = "Yes" if p_val < 0.05 else "No"
    print(f"{name:<25}  {len(lo):>5}/{len(hi):<5}  {t_val:>6.2f}  {p_val:>8.3f}  {sig}")


In [ ]:
# ── 2.2  Boxplots: original · with outliers · after IQR removal ─────────────
def iqr_remove(arr):
    q1, q3 = np.percentile(arr, [25, 75])
    fence   = 1.5 * (q3 - q1)
    return arr[(arr >= q1 - fence) & (arr <= q3 + fence)]

scenarios = [
    ("Elderly added", base_low, base_high, base_low, high_with_elderly),
    ("Sensor errors added", base_low, base_high, low_with_sensor, base_high),
    ("Both added", base_low, base_high, low_with_sensor, high_with_elderly),
]

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
np.random.seed(SEED)

for row_i, (scenario, orig_lo, orig_hi, cont_lo, cont_hi) in enumerate(scenarios):
    lo_iqr = iqr_remove(cont_lo)
    hi_iqr = iqr_remove(cont_hi)

    _, p_orig = stats.ttest_ind(orig_lo, orig_hi)
    _, p_cont = stats.ttest_ind(cont_lo, cont_hi)
    _, p_iqr  = stats.ttest_ind(lo_iqr,  hi_iqr)

    for col_j, (title, l, h, p_val) in enumerate([
        ("Original",                       orig_lo, orig_hi, p_orig),
        (f"With outliers\n({scenario})",   cont_lo, cont_hi, p_cont),
        ("After IQR removal",              lo_iqr,  hi_iqr,  p_iqr),
    ]):
        ax = axes[row_i, col_j]
        bp = ax.boxplot([l, h], labels=["Low", "High"], patch_artist=True,
                        medianprops={"color": "black", "lw": 2})
        bp["boxes"][0].set_facecolor("steelblue")
        bp["boxes"][1].set_facecolor("tomato")
        for k, arr in enumerate([l, h]):
            jitter = np.random.uniform(-0.07, 0.07, len(arr))
            ax.scatter(np.full(len(arr), k + 1) + jitter,
                       arr, alpha=0.5, s=18, zorder=3,
                       color=["steelblue", "tomato"][k])
        stars = ("***" if p_val < 0.001 else
                 "**"  if p_val < 0.01  else
                 "*"   if p_val < 0.05  else "ns")
        ax.set_title(f"{title}\np={p_val:.3f}  {stars}", fontsize=9)
        ax.set_ylabel("Mean HR (BPM)")

plt.suptitle("Impact of different outlier types — low vs high stress comparison",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

### Detecting outliers

An outlier is a value far from the rest of the data. "Far" needs a definition. We will discuss 3 different ones in this section:
1. IQR: already defined and used above. This method uses use **quartiles**. Sort the data and split it into four equal parts: Q1 is the value below which 25% of the data falls, Q2 is the median (50%), Q3 is the 75% point. The **interquartile range (IQR)** is the the width of the middle half of the data: `IQR = Q3 − Q1`. Flag anything below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`. This is the rule behind boxplot whiskers, so any dot plotted separately is an IQR outlier.
2. Z-score: How many standard deviations from the mean is a data point: `z = (x − mean) / sd`.
3. Modified z-score: Fixes the z-score's blind spot by swapping in statistics that outliers can't distort. Instead of the mean, use the median. Instead of the standard deviation, use the MAD — the median distance from the median, `MAD = median(|x − median|)`.

In [ ]:
# ── 2.3  Outlier detection methods compared ─────────────────────────────
from scipy.stats import zscore

def report_outliers(arr, label):
    q1, q3   = np.percentile(arr, [25, 75])
    iqr_mask = (arr < q1 - 1.5*(q3-q1)) | (arr > q3 + 1.5*(q3-q1))

    # CODE HERE START
    # 1. Z-score: distance from the mean in SDs. Flag |z| > 3.
    # 2. Modified z-score: 0.6745 * (x - median) / MAD, where
    #    MAD = median(|x - median|). Flag |mz| > 3.5.

    # z_mask   =
    # mz_mask  =

    # CODE HERE END

    print(f"\n{label}  (n={len(arr)})")
    for method, mask in [("IQR",              iqr_mask),
                         ("Z-score |z|>3",    z_mask),
                         ("Modified z-score", mz_mask)]:
        vals = ", ".join(f"{v:.1f}" for v in arr[mask]) if mask.any() else "none"
        print(f"  {method:<22}  {mask.sum()} flagged  [{vals}]")

report_outliers(high_with_elderly, "High-stress + elderly participants")
report_outliers(low_with_sensor,   "Low-stress + sensor errors")


**Q2 —**

a. Change `N_ELDERLY` and `N_SENSOR`. At what value does the addition of outliers participants flip the t-test from significant to not significant? Does your answer change with `N_SAMPLES`?  
b. How many of the three elderly participants does each method flag? Should we remove them? What does removing them say about which population our results apply to?  
c. The sensor values (200–220 BPM) are physiologically impossible during an exam. Is removing those a different decision from removing the elderly participants? Explain the distinction.  
d. Looking at the two detection methods in cell 2.3, which is most appropriate when the outliers are sensor errors?

> *Your answer here*
